# 0630 multi-kernel smoke plotting

Single-timer CGT smoke; WD/cross-timer divergence is skipped by design.

In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATE_BASE = "20260630"
RUN_ID = "20260630-kernels-smoke-dsub"
OUTPUT_ROOT = "output_kernels_smoke_0630_dsub"
HOST_ORDER = ["cgnr6760pn2", "camd9554n1", "c920bn3"]
DATA_ROOT = Path(f"/astrum/home/hpchzy/code/data/{DATE_BASE}")
OUT_DIR = Path("stencil/plots_0630_kernels_smoke_dsub")
OUT_DIR.mkdir(parents=True, exist_ok=True)
KERNEL_LABELS = {
    "openblas_gemm":"OpenBLAS GEMM", "openblas_gemv":"OpenBLAS GEMV", "openblas_dot":"OpenBLAS DOT", "openblas_axpy":"OpenBLAS AXPY",
    "stream_triad":"STREAM Triad", "hpcg_spmv":"HPCG SpMV", "npb_ft_fft":"NPB-FT FFT", "npb_ep":"NPB-EP",
}

def read_tm_dir(d):
    vals=[]
    for f in sorted(Path(d).glob("*.csv")):
        arr=np.loadtxt(f, delimiter=",", usecols=[1], ndmin=1)
        vals.extend(np.ravel(arr).astype(float))
    return np.asarray(vals, dtype=float)

def read_tf_dir(d):
    vals=[]
    for f in sorted(Path(d).glob("*.csv")):
        arr=np.loadtxt(f, delimiter=",", usecols=[2], ndmin=1)
        vals.extend(np.ravel(arr).astype(float))
    return np.asarray(vals, dtype=float)

def read_hist_cdf(f):
    arr=np.loadtxt(f, delimiter=",", ndmin=2)
    if arr.shape[1] < 2:
        return np.array([]), np.array([])
    x=arr[:,0].astype(float); p=arr[:,1].astype(float)
    order=np.argsort(x); x=x[order]; p=p[order]
    c=np.cumsum(p)
    if c.size and c[-1] != 0: c=c/c[-1]
    return x,c

rows=[]
for host in HOST_ORDER:
    root = DATA_ROOT / host / OUTPUT_ROOT / RUN_ID / "shuffle0"
    if not root.exists():
        raise FileNotFoundError(root)
    for filt in sorted(root.glob("*_filt")):
        m=re.match(r"(?P<kernel>.+)_cgt_np(?P<np>\d+)_size(?P<size>\d+)_nsampRatio(?P<ratio>[\d.]+)_nsamp(?P<nsamp>\d+)_filt", filt.name)
        if not m: continue
        gd=m.groupdict(); kernel=gd["kernel"]; size=int(gd["size"]); nsamp=int(gd["nsamp"])
        tm_dir=root / f"{kernel}_cgt_np{gd['np']}_size{size}"
        tf_dir=root / f"{kernel}_cgt_np{gd['np']}_size{size}_nsampRatio{gd['ratio']}_nsamp{nsamp}_tf"
        tm=read_tm_dir(tm_dir); tf=read_tf_dir(tf_dir)
        tx,tc=read_hist_cdf(filt / "tr_hist.csv")
        tr_median=float(tx[np.searchsorted(tc,0.5)]) if len(tx) else np.nan
        rows.append(dict(host=host,kernel=kernel,size=size,nsamp=nsamp,timer="cgt",tm_median=float(np.median(tm)),tf_median=float(np.median(tf)),tr_median=tr_median,tm_n=len(tm),tf_n=len(tf)))
summary=pd.DataFrame(rows).sort_values(["kernel","host"])
summary.to_csv(OUT_DIR / "kernels_smoke_summary.csv", index=False)
print(summary)

# one compact median summary figure
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for ax, host in zip(axes, HOST_ORDER):
    sub=summary[summary.host==host].copy()
    labels=[KERNEL_LABELS.get(k,k) for k in sub.kernel]
    x=np.arange(len(sub))
    ax.plot(x, sub.tm_median, marker='o', label='TM median')
    ax.plot(x, sub.tr_median, marker='s', label='TR median')
    ax.set_title(host)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('ns')
    ax.grid(True, alpha=0.3)
axes[0].legend()
fig.suptitle('0630 multi-kernel smoke: CGT TM/TR medians')
fig.tight_layout()
fig.savefig(OUT_DIR / 'kernels_smoke_tm_tr_median_by_host.png', dpi=180)
plt.close(fig)

# per-kernel CDFs, three host panels, TM raw CDF + TR hist CDF
for kernel in summary.kernel.unique():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, host in zip(axes, HOST_ORDER):
        row=summary[(summary.kernel==kernel)&(summary.host==host)].iloc[0]
        root = DATA_ROOT / host / OUTPUT_ROOT / RUN_ID / "shuffle0"
        tm_dir=root / f"{kernel}_cgt_np64_size{int(row["size"])}"
        filt=next(root.glob(f"{kernel}_cgt_np64_size{int(row["size"])}_nsampRatio0.5_nsamp*_filt"))
        tm=np.sort(read_tm_dir(tm_dir)); y=np.arange(1,len(tm)+1)/len(tm)
        ax.plot(tm,y,label='TM raw CDF')
        tx,tc=read_hist_cdf(filt / 'tr_hist.csv')
        if len(tx): ax.plot(tx,tc,label='TR hist CDF')
        ax.set_title(f"{host} size={int(row["size"])}")
        ax.set_xlabel('ns'); ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('CDF')
    axes[0].legend()
    fig.suptitle(f"{KERNEL_LABELS.get(kernel,kernel)} smoke CGT TM/TR CDF")
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{kernel}_smoke_cgt_tm_tr_cdf.png", dpi=180)
    plt.close(fig)
print('single timer smoke: WD/cross-timer divergence skipped by design')
print('wrote', OUT_DIR)
